# CNN vs CNN-Transformer — CICIDS2018 Full Dataset (Kaggle T4 x2)

This notebook trains **two** models on the **full CIC-IDS-2018** dataset (~7 GB, 10 CSV files):

1. **CNN Classifier** — Conv1D + Global Average Pooling
2. **CNN-Transformer** — Conv1D tokenizer + Transformer encoder

**Hardware target:** Kaggle T4 × 2 (15 GB VRAM each), 30 GB system RAM  
**Split:** 70 / 10 / 20 (train / validation / test)  
**Dataset files (inside `CICIDS2018/`):**

| File | Day |
|------|-----|
| `Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv` | Wed 14 Feb |
| `Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv` | Thu 15 Feb |
| `Friday-16-02-2018_TrafficForML_CICFlowMeter.csv` | Fri 16 Feb |
| `Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv` | Tue 20 Feb |
| `Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv` | Wed 21 Feb |
| `Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv` | Thu 22 Feb |
| `Friday-23-02-2018_TrafficForML_CICFlowMeter.csv` | Fri 23 Feb |
| `Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv` | Wed 28 Feb |
| `Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv` | Thu 01 Mar |
| `Friday-02-03-2018_TrafficForML_CICFlowMeter.csv` | Fri 02 Mar |

Outputs per model:
- Model checkpoint (`.pth`)
- Preprocessing artifacts (`.pkl`)
- ROC-AUC curve plot
- Side-by-side metric comparison

In [ ]:
# Cell 1: Setup & install
import os, sys, glob, warnings, importlib, gc
from importlib.metadata import version, PackageNotFoundError

warnings.filterwarnings('ignore')
os.environ['TORCHDYNAMO_DISABLE'] = '1'

REPO_URL = 'https://github.com/samaraweeramethun-eng/CNN-Transformer.git'
REPO_DIR = '/kaggle/working/cnn_transformer_only'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

!pip install -q joblib psutil scikit-learn
!pip install -q -e .

SRC_DIR = os.path.join(REPO_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

for name in list(sys.modules):
    if name == 'cnn_transformer_only' or name.startswith('cnn_transformer_only.'):
        del sys.modules[name]

import torch
cnn_transformer_only = importlib.import_module('cnn_transformer_only')

print('module file:', cnn_transformer_only.__file__)
try:
    pkg_ver = version('cnn-transformer-only')
except PackageNotFoundError:
    pkg_ver = getattr(cnn_transformer_only, '__version__', 'unknown')

print('✓ cnn_transformer_only version:', pkg_ver)
print('✓ PyTorch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
# Cell 2: Locate CICIDS2018 dataset folder
import pandas as pd

# === SET THIS if auto-detection fails ===
CICIDS2018_DIR = ''  # e.g. '/kaggle/input/cicids2018/CICIDS2018'

if not CICIDS2018_DIR or not os.path.isdir(CICIDS2018_DIR):
    # Auto-detect: look for the characteristic filenames under /kaggle/input
    candidates = sorted(glob.glob('/kaggle/input/**/CICIDS2018', recursive=True))
    if not candidates:
        candidates = sorted(glob.glob('/kaggle/input/**/*TrafficForML*CICFlowMeter*.csv', recursive=True))
        if candidates:
            candidate_dirs = list(set(os.path.dirname(c) for c in candidates))
            candidate_dirs.sort(key=lambda d: len(glob.glob(os.path.join(d, '*.csv'))), reverse=True)
            CICIDS2018_DIR = candidate_dirs[0]
    else:
        CICIDS2018_DIR = candidates[0]

if not CICIDS2018_DIR or not os.path.isdir(CICIDS2018_DIR):
    # Last resort: any folder with many CSVs
    all_csvs = sorted(glob.glob('/kaggle/input/**/*.csv', recursive=True))
    if all_csvs:
        from collections import Counter
        dir_counts = Counter(os.path.dirname(c) for c in all_csvs)
        CICIDS2018_DIR = dir_counts.most_common(1)[0][0]

DATA_PATH = CICIDS2018_DIR
csv_files = sorted(glob.glob(os.path.join(DATA_PATH, '*.csv')))

print(f'Dataset folder: {DATA_PATH}')
print(f'CSV files found: {len(csv_files)}')
total_size_gb = sum(os.path.getsize(f) for f in csv_files) / 1024**3
print(f'Total size: {total_size_gb:.2f} GB')
for f in csv_files:
    print(f'  {os.path.basename(f):60s} {os.path.getsize(f)/1024**2:8.1f} MB')

# Quick peek at columns
peek = pd.read_csv(csv_files[0], nrows=3)
label_candidates = [c for c in peek.columns if 'label' in c.lower()]
print(f'\nColumns: {len(peek.columns)} | Label col: {label_candidates[0] if label_candidates else "NOT FOUND"}')

In [ ]:
# Cell 3: Configure — optimised for 2×T4 + 30 GB RAM + ~7 GB dataset
from cnn_transformer_only.config import CNNTransformerConfig

cfg = CNNTransformerConfig(
    input_path      = DATA_PATH,
    output_dir      = '/kaggle/working/artifacts_2018',
    csv_chunksize   = 150_000,       # smaller chunks → lower peak RAM
    max_rows        = 0,             # 0 = load ALL rows
    val_size        = 0.10,          # 10 % validation
    test_size       = 0.20,          # 20 % test  →  70 / 10 / 20
    epochs          = 25,
    batch_size      = 16384,         # model is tiny (~100K params), need huge batches
    val_batch_size  = 32768,         # eval can use even larger batches
    lr              = 3e-3,          # scale LR with larger batch (linear scaling)
    weight_decay    = 1e-4,
    label_smoothing = 0.05,
    conv_channels   = 96,
    num_layers      = 3,
    num_heads       = 8,
    d_model         = 192,
    d_ff            = 768,
    dropout         = 0.2,
    cnn_fc_dim      = 128,
    undersampling_ratio = 0.15,
    num_workers     = 4,             # more workers to keep GPUs fed
    # interpretability params (unused in this notebook but kept for compat)
    ig_steps        = 32,
    ig_samples      = 512,
)

print(f'Split: 70/10/20  (val_size={cfg.val_size}, test_size={cfg.test_size})')
print(f'Epochs: {cfg.epochs} | batch: {cfg.batch_size} | lr: {cfg.lr}')
print(f'd_model: {cfg.d_model} | conv_channels: {cfg.conv_channels} | layers: {cfg.num_layers}')
print(f'Output dir: {cfg.output_dir}')

---
## Shared Data Loading

Load the full dataset **once** into a memory-efficient numpy array, then split.  
Both models share the exact same split for fair comparison.

In [ ]:
# Cell 4: Load full CICIDS2018 dataset (chunked, memory-efficient)
import time, psutil
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from cnn_transformer_only.data import (
    IntelligentDataBalancer,
    load_cicids_feature_matrix,
)

ram = psutil.virtual_memory()
print(f'RAM before load: {ram.used/1024**3:.1f} / {ram.total/1024**3:.1f} GB')

t0 = time.time()
X_raw, y_raw, feature_cols, label_col = load_cicids_feature_matrix(
    cfg.input_path,
    max_rows=cfg.max_rows,
    chunksize=cfg.csv_chunksize,
)
print(f'Loaded {len(y_raw):,} rows × {len(feature_cols)} features in {time.time()-t0:.0f}s')
print(f'Label col: {label_col}')
print(f'Class distribution: benign={int((y_raw==0).sum()):,}  attack={int((y_raw==1).sum()):,}  '
      f'({100*y_raw.mean():.2f}% attack)')

# ── Feature filtering: remove leaky, zero-variance, and redundant columns ──
# 1) Leaky identifiers — model memorises ports/protocol instead of behaviour
LEAKY = {'Dst Port', 'Src Port', 'Protocol'}

# 2) Near-zero-variance columns (mostly zeros across CICIDS2018)
ZERO_VAR = {
    'Bwd PSH Flags', 'Bwd URG Flags', 'Fwd URG Flags',
    'CWE Flag Count',
    'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg',
    'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg',
}

# 3) Redundant / highly correlated (subflow == total in this dataset)
REDUNDANT = {
    'Subflow Fwd Pkts', 'Subflow Fwd Byts',
    'Subflow Bwd Pkts', 'Subflow Bwd Byts',
    'Fwd Header Len',
}

DROP_SET = LEAKY | ZERO_VAR | REDUNDANT
# Match case-insensitively against actual column names
drop_map = {}
for col in feature_cols:
    for d in DROP_SET:
        if col.strip().lower() == d.lower():
            drop_map[col] = d
            break

drop_idxs = sorted([feature_cols.index(c) for c in drop_map])
if drop_idxs:
    X_raw = np.delete(X_raw, drop_idxs, axis=1)
    kept = [c for c in feature_cols if c not in drop_map]
    print(f'\nDropped {len(drop_idxs)} features (leaky/zero-var/redundant):')
    for col, reason in sorted(drop_map.items()):
        tag = 'LEAKY' if reason in LEAKY else ('ZERO-VAR' if reason in ZERO_VAR else 'REDUNDANT')
        print(f'  [{tag:9s}] {col}')
    feature_cols = kept
    print(f'Remaining features: {len(feature_cols)}')
else:
    print('No matching columns found to drop (column names may differ)')

ram = psutil.virtual_memory()
print(f'RAM after load+filter: {ram.used/1024**3:.1f} / {ram.total/1024**3:.1f} GB')

In [ ]:
# Cell 5: 70/10/20 stratified split + StandardScaler + class balancing
import gc

holdout_ratio = cfg.val_size + cfg.test_size   # 0.30
test_fraction = cfg.test_size / holdout_ratio   # 0.20/0.30 ≈ 0.667

X_train_raw, X_holdout, y_train, y_holdout = train_test_split(
    X_raw, y_raw,
    test_size=holdout_ratio,
    stratify=y_raw,
    random_state=cfg.random_state,
)
del X_raw, y_raw; gc.collect()

X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_holdout, y_holdout,
    test_size=test_fraction,
    stratify=y_holdout,
    random_state=cfg.random_state,
)
del X_holdout, y_holdout; gc.collect()

print(f'Train: {len(y_train):>10,}  (attack {100*y_train.mean():.2f}%)')
print(f'Val:   {len(y_val):>10,}  (attack {100*y_val.mean():.2f}%)')
print(f'Test:  {len(y_test):>10,}  (attack {100*y_test.mean():.2f}%)')

# Fit scaler on train only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw).astype(np.float32)
del X_train_raw; gc.collect()

X_val_scaled = scaler.transform(X_val_raw).astype(np.float32)
del X_val_raw; gc.collect()

X_test_scaled = scaler.transform(X_test_raw).astype(np.float32)
del X_test_raw; gc.collect()

# Balance training set
balancer = IntelligentDataBalancer(cfg.undersampling_ratio, cfg.random_state)
X_train_bal, y_train_bal = balancer.balance_classes(X_train_scaled, y_train)
del X_train_scaled, y_train; gc.collect()

input_dim = X_train_bal.shape[1]
print(f'\nBalanced train: {len(y_train_bal):,}  (attack {100*y_train_bal.mean():.2f}%)')
print(f'Input dim: {input_dim}')

ram = psutil.virtual_memory()
print(f'RAM after split: {ram.used/1024**3:.1f} / {ram.total/1024**3:.1f} GB')

In [ ]:
# Cell 6: Build shared DataLoaders
from cnn_transformer_only.data import build_dataloaders
from cnn_transformer_only.utils.device import setup_device
from torch.utils.data import DataLoader, TensorDataset

device, multi_gpu = setup_device()

# Enable cuDNN autotuner — free speedup for fixed-size inputs
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

# Model is very small (~100K params) → need massive batches to saturate GPU
batch_size = cfg.batch_size           # 16384 total → 8192/GPU with DataParallel
val_batch_size = cfg.val_batch_size   # 32768 total → 16384/GPU

train_loader, val_loader, _ = build_dataloaders(
    X_train_bal, y_train_bal,
    X_val_scaled, y_val,
    batch_size=batch_size,
    val_batch_size=val_batch_size,
    num_workers=cfg.num_workers,
)

test_dataset = TensorDataset(torch.FloatTensor(X_test_scaled), torch.LongTensor(y_test))
test_loader = DataLoader(
    test_dataset,
    batch_size=val_batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=cfg.num_workers > 0,
    prefetch_factor=4 if cfg.num_workers > 0 else None,
)

print(f'Device: {device} | Multi-GPU: {multi_gpu}')
print(f'Train loader: {len(train_loader.dataset):,} samples, {len(train_loader)} batches')
print(f'Val loader:   {len(val_loader.dataset):,} samples')
print(f'Test loader:  {len(test_loader.dataset):,} samples')

---
## Training Helpers

In [ ]:
# Cell 7: Training & evaluation functions (shared by both models)
# Uses AMP (automatic mixed precision) — T4 tensor cores give ~2× speedup on FP16
import random
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.parallel import DataParallel
from torch.amp import autocast, GradScaler
from cnn_transformer_only.data import (
    calculate_comprehensive_metrics,
    binary_predictions_from_proba,
    find_best_f1_threshold,
)

# AMP scaler for mixed precision training
amp_scaler = GradScaler('cuda') if torch.cuda.is_available() else None
USE_AMP = torch.cuda.is_available()

def set_seeds(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def train_epoch(model, loader, criterion, optimizer, scheduler, device):
    model.train()
    running_loss = 0.0
    for data, target in loader:
        data = data.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=USE_AMP):
            logits = model(data)
            loss = criterion(logits, target)
        if USE_AMP:
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            amp_scaler.step(optimizer)
            amp_scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        if scheduler is not None:
            scheduler.step()
        running_loss += loss.item()
    return running_loss / max(len(loader), 1)

def eval_epoch(model, loader, criterion, device):
    model.eval()
    losses, all_probs, all_targets = [], [], []
    with torch.no_grad(), autocast('cuda', enabled=USE_AMP):
        for data, target in loader:
            data = data.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            logits = model(data)
            loss = criterion(logits, target)
            losses.append(loss.item())
            probs = F.softmax(logits, dim=1)[:, 1]
            all_probs.append(probs.float().cpu().numpy())
            all_targets.append(target.cpu().numpy())
    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_targets)
    y_pred = (y_prob >= 0.5).astype(np.int64)
    metrics = calculate_comprehensive_metrics(y_true, y_pred, y_prob)
    return float(np.mean(losses)), metrics, y_prob, y_true

def eval_with_threshold(model, loader, criterion, device, threshold):
    model.eval()
    losses, all_probs, all_targets = [], [], []

    with torch.no_grad(), autocast('cuda', enabled=USE_AMP):
        for data, target in loader:
            data = data.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)

            logits = model(data)
            loss = criterion(logits, target)
            losses.append(loss.item())

            probs = F.softmax(logits, dim=1)[:, 1]
            all_probs.append(probs.float().cpu().numpy())
            all_targets.append(target.cpu().numpy())

    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_targets)
    y_pred = binary_predictions_from_proba(y_prob, threshold)
    metrics = calculate_comprehensive_metrics(y_true, y_pred, y_prob)
    return float(np.mean(losses)), metrics, y_prob, y_true


def train_and_evaluate(
    model,
    train_ldr,
    val_ldr,
    test_ldr,
    cfg,
    device,
    feature_cols,
    model_name='Model',
    multi_gpu=False,
):
    """Generic training loop with AMP. Returns (checkpoint_dict, test_probs, test_targets)."""
    global amp_scaler
    set_seeds(cfg.random_state)

    # Reset AMP scaler for each model to avoid stale state
    if USE_AMP:
        amp_scaler = GradScaler('cuda')

    if multi_gpu:
        model = DataParallel(model)

    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = (
        optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=cfg.lr,
            epochs=cfg.epochs,
            steps_per_epoch=max(len(train_ldr), 1),
        )
        if len(train_ldr) > 0
        else None
    )

    best_auc = 0.0
    best_state = None

    for epoch in range(1, cfg.epochs + 1):
        t_loss = train_epoch(model, train_ldr, criterion, optimizer, scheduler, device)
        v_loss, v_met, _, _ = eval_epoch(model, val_ldr, criterion, device)
        print(
            f'  [{model_name}] Epoch {epoch:02d} | TrLoss {t_loss:.4f} | '
            f'VaLoss {v_loss:.4f} | ROC-AUC {v_met["auc_roc"]:.4f} | F1 {v_met["f1_score"]:.4f}'
        )

        if v_met['auc_roc'] > best_auc:
            best_auc = v_met['auc_roc']
            sd = model.module.state_dict() if isinstance(model, DataParallel) else model.state_dict()
            best_state = {
                'model_state_dict': sd,
                'metrics': v_met,
                'config': cfg.__dict__,
                'feature_columns': feature_cols,
                'model_type': model_name,
            }

    if best_state is None:
        print(f'{model_name}: training did not improve.')
        return None, None, None

    # Restore best weights
    base_model = model.module if isinstance(model, DataParallel) else model
    base_model.load_state_dict(best_state['model_state_dict'])

    # Tune threshold on val set
    _, _, val_probs, val_targets = eval_epoch(model, val_ldr, criterion, device)
    best_thr, best_f1 = find_best_f1_threshold(val_targets, val_probs)
    best_state['best_threshold'] = best_thr
    best_state['best_val_f1_at_threshold'] = best_f1
    print(f'  [{model_name}] Best threshold: {best_thr:.4f} (val F1 {best_f1:.4f})')

    # Evaluate on test set
    t_loss, t_met, t_probs, t_targets = eval_with_threshold(
        model, test_ldr, criterion, device, threshold=best_thr
    )
    best_state['test_metrics'] = t_met

    print(f'\n  {"="*55}')
    print(f'  {model_name} — TEST SET RESULTS')
    print(f'  {"="*55}')
    print(f'    Loss:      {t_loss:.4f}')
    print(f'    Threshold: {best_thr:.4f}')
    print(f'    ROC-AUC:   {t_met["auc_roc"]:.4f}')
    print(f'    PR-AUC:    {t_met["auc_pr"]:.4f}')
    print(f'    F1-Score:  {t_met["f1_score"]:.4f}')
    print(f'    Precision: {t_met["precision"]:.4f}')
    print(f'    Recall:    {t_met["recall"]:.4f}')
    print(f'    Accuracy:  {t_met["accuracy"]:.4f}')
    print(f'  {"="*55}\n')

    return best_state, t_probs, t_targets


print('Training helpers ready.')

---
## Part 1 — CNN Classifier

In [ ]:
# Cell 8: Train CNN Classifier
from cnn_transformer_only.models.cnn_classifier import CNNClassifier
import joblib

os.makedirs(cfg.output_dir, exist_ok=True)

ram = psutil.virtual_memory()
print(f'RAM: {ram.used/1024**3:.1f} / {ram.total/1024**3:.1f} GB')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f'GPU {i} VRAM: {(total-free)/1024**3:.1f} / {total/1024**3:.1f} GB')

cnn_model = CNNClassifier(
    input_dim=input_dim,
    conv_channels=cfg.conv_channels,
    fc_dim=cfg.cnn_fc_dim,
    dropout=cfg.dropout,
).to(device)

print(f'CNN Classifier params: {sum(p.numel() for p in cnn_model.parameters()):,}')

t0 = time.time()
cnn_state, cnn_test_probs, cnn_test_targets = train_and_evaluate(
    cnn_model, train_loader, val_loader, test_loader, cfg, device, feature_cols,
    model_name='CNN_Classifier', multi_gpu=multi_gpu,
)
print(f'CNN Classifier training time: {(time.time()-t0)/60:.1f} min')

if cnn_state is not None:
    cnn_path = os.path.join(cfg.output_dir, 'cnn_classifier.pth')
    torch.save(cnn_state, cnn_path)
    print(f'Saved: {cnn_path}')

# Free GPU memory before next model
del cnn_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

---
## Part 2 — CNN-Transformer

In [ ]:
# Cell 9: Train CNN-Transformer
# CNN-Transformer needs much more VRAM per sample (attention + FFN) than CNN Classifier.
# Rebuild loaders with smaller batch to fit in 15 GB/GPU.
from cnn_transformer_only.models.cnn_transformer import CNNTransformerIDS

CT_BATCH = 2048        # 1024/GPU — Transformer attention is O(seq_len² × d_model)
CT_VAL_BATCH = 4096    # eval uses less memory (no grads)

ct_train_loader, ct_val_loader, _ = build_dataloaders(
    X_train_bal, y_train_bal,
    X_val_scaled, y_val,
    batch_size=CT_BATCH,
    val_batch_size=CT_VAL_BATCH,
    num_workers=cfg.num_workers,
)
ct_test_loader = DataLoader(
    test_dataset,
    batch_size=CT_VAL_BATCH,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
    persistent_workers=cfg.num_workers > 0,
)

ram = psutil.virtual_memory()
print(f'RAM: {ram.used/1024**3:.1f} / {ram.total/1024**3:.1f} GB')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f'GPU {i} VRAM: {(total-free)/1024**3:.1f} / {total/1024**3:.1f} GB')

ct_model = CNNTransformerIDS(
    input_dim=input_dim,
    d_model=cfg.d_model,
    conv_channels=cfg.conv_channels,
    num_layers=cfg.num_layers,
    num_heads=cfg.num_heads,
    d_ff=cfg.d_ff,
    dropout=cfg.dropout,
).to(device)

print(f'CNN-Transformer params: {sum(p.numel() for p in ct_model.parameters()):,}')
print(f'Batch size: {CT_BATCH} (vs {cfg.batch_size} for CNN)')

# Scale LR down proportionally: batch 2048 vs 16384 → LR / 8
from copy import copy
cfg_ct = copy(cfg)
cfg_ct.lr = cfg.lr * (CT_BATCH / cfg.batch_size)  # 3e-3 * 2048/16384 = 3.75e-4
print(f'LR scaled: {cfg_ct.lr:.6f}')

t0 = time.time()
ct_state, ct_test_probs, ct_test_targets = train_and_evaluate(
    ct_model, ct_train_loader, ct_val_loader, ct_test_loader, cfg_ct, device, feature_cols,
    model_name='CNN_Transformer', multi_gpu=multi_gpu,
)
print(f'CNN-Transformer training time: {(time.time()-t0)/60:.1f} min')

if ct_state is not None:
    ct_path = os.path.join(cfg.output_dir, 'cnn_transformer_ids.pth')
    torch.save(ct_state, ct_path)
    print(f'Saved: {ct_path}')

del ct_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

---
## Part 3 — Metrics & ROC-AUC Curves

In [ ]:
# Cell 10: Test metrics comparison table
import pandas as pd

cnn_metrics  = cnn_state.get('test_metrics', {}) if cnn_state else {}
ct_metrics   = ct_state.get('test_metrics', {})  if ct_state  else {}

metric_keys = ['accuracy', 'auc_roc', 'auc_pr', 'f1_score', 'precision', 'recall']

comparison = pd.DataFrame({
    'Metric': metric_keys,
    'CNN Classifier': [cnn_metrics.get(k, None) for k in metric_keys],
    'CNN-Transformer': [ct_metrics.get(k, None) for k in metric_keys],
}).set_index('Metric')

comparison['Δ (Transformer − CNN)'] = comparison['CNN-Transformer'] - comparison['CNN Classifier']

print('=' * 65)
print('MODEL COMPARISON — CICIDS2018 Test Set (20%)')
print('=' * 65)
display(comparison)

print('\nPer-metric winner:')
for k in metric_keys:
    c = cnn_metrics.get(k, 0)
    t = ct_metrics.get(k, 0)
    w = 'CNN-Transformer' if t > c else ('CNN Classifier' if c > t else 'Tie')
    print(f'  {k:12s}: {w} ({t:.4f} vs {c:.4f})')

In [ ]:
# Cell 11: ROC-AUC curves for both models
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

results = [
    ('CNN Classifier',  cnn_test_probs, cnn_test_targets, cnn_metrics),
    ('CNN-Transformer', ct_test_probs,  ct_test_targets,  ct_metrics),
]

colors = ['#2196F3', '#FF5722']

# --- Subplot 1: Individual ROC curves ---
for idx, (name, probs, targets, met) in enumerate(results):
    if probs is None or targets is None:
        continue
    fpr, tpr, _ = roc_curve(targets, probs)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=colors[idx], lw=2,
                 label=f'{name} (AUC = {roc_auc:.4f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random (AUC = 0.5)')
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curves — CICIDS2018 Test Set', fontsize=14)
axes[0].legend(loc='lower right', fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([-0.01, 1.01])
axes[0].set_ylim([-0.01, 1.01])

# --- Subplot 2: Bar chart of all metrics ---
x_pos = np.arange(len(metric_keys))
width = 0.35
cnn_vals = [cnn_metrics.get(k, 0) for k in metric_keys]
ct_vals  = [ct_metrics.get(k, 0)  for k in metric_keys]

bars1 = axes[1].bar(x_pos - width/2, cnn_vals, width, label='CNN Classifier', color=colors[0], alpha=0.85)
bars2 = axes[1].bar(x_pos + width/2, ct_vals,  width, label='CNN-Transformer', color=colors[1], alpha=0.85)

axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(metric_keys, rotation=25, ha='right')
axes[1].set_ylim(0, 1.08)
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('Metrics Comparison — CICIDS2018 Test Set', fontsize=14)
axes[1].legend(loc='lower right', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

for bar in bars1:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=7)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
fig_path = os.path.join(cfg.output_dir, 'roc_auc_and_comparison_cicids2018.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved plot: {fig_path}')

In [ ]:
# Cell 12: Precision-Recall curves (supplementary)
from sklearn.metrics import precision_recall_curve, average_precision_score

fig, ax = plt.subplots(figsize=(8, 6))

for idx, (name, probs, targets, met) in enumerate(results):
    if probs is None or targets is None:
        continue
    prec, rec, _ = precision_recall_curve(targets, probs)
    ap = average_precision_score(targets, probs)
    ax.plot(rec, prec, color=colors[idx], lw=2,
            label=f'{name} (AP = {ap:.4f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves — CICIDS2018 Test Set', fontsize=14)
ax.legend(loc='lower left', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.05])

plt.tight_layout()
pr_path = os.path.join(cfg.output_dir, 'precision_recall_cicids2018.png')
plt.savefig(pr_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {pr_path}')

In [ ]:
# Cell 13: Final summary
print('\n' + '='*65)
print('CICIDS2018 FULL DATASET — TRAINING COMPLETE')
print('='*65)
print(f'Dataset:     {DATA_PATH}')
print(f'Total rows:  loaded from {len(csv_files)} CSV files (~{total_size_gb:.1f} GB)')
print(f'Split:       70% train / 10% val / 20% test')
print(f'Artifacts:   {cfg.output_dir}')

# List saved artifacts
saved = sorted(glob.glob(os.path.join(cfg.output_dir, '*')))
print(f'\nSaved files ({len(saved)}):')
for f in saved:
    print(f'  {os.path.basename(f):50s} {os.path.getsize(f)/1024**2:.1f} MB')

ram = psutil.virtual_memory()
print(f'\nFinal RAM: {ram.used/1024**3:.1f} / {ram.total/1024**3:.1f} GB')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f'GPU {i}: {(total-free)/1024**3:.1f} / {total/1024**3:.1f} GB used')